# Qdrant

## kill process

```
# Stop Qdrant if running
pkill -f qdrant

# Verify
curl http://localhost:6333/collections/data_ds_bsc
```

## Create Qdrant Instance from .sqlite on Server

### Fully new

```
# 1. Check what's currently in your storage
ls ~/qdrant_storage/collections/

# 2. Stop Qdrant if running
pkill -f qdrant
sleep 2

# 3. Replace the old sqlite with DE's new one
# (assuming DE sent you a new zip or sqlite file)
# If zip:
unzip ~/new_qdrant_data.zip -d ~/qdrant_storage_new
# Copy the new sqlite to the right place
cp ~/qdrant_storage_new/path/to/storage.sqlite \
   ~/qdrant_storage/collections/university-docs/storage.sqlite

# 4. Start Qdrant
nohup ~/qdrant --config-path ~/qdrant_config/config.yaml \
  > ~/qdrant.log 2>&1 &
sleep 3
curl http://localhost:6333/collections

# 5. If collection is empty, re-upload from sqlite
python3 -c "
import sqlite3, pickle
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance,
    SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector
)

client = QdrantClient(host='localhost', port=6333)
conn = sqlite3.connect(
    '/home/user1/qdrant_storage/collections/university-docs/storage.sqlite'
)
rows = conn.execute('SELECT id, point FROM points').fetchall()
conn.close()
print(f'Found {len(rows)} points')

client.recreate_collection(
    collection_name='data_ds_bsc',
    vectors_config={'': VectorParams(size=1024, distance=Distance.COSINE)},
    sparse_vectors_config={
        'sparse': SparseVectorParams(index=SparseIndexParams())
    }
)

points = []
for row_id, blob in rows:
    point = pickle.loads(blob)
    dense = point.vector.get('', [])
    sparse_raw = point.vector.get('sparse', None)
    sparse = None
    if sparse_raw:
        sparse = SparseVector(
            indices=sparse_raw.indices,
            values=sparse_raw.values,
        )
    points.append(PointStruct(
        id=point.id,
        vector={'': dense, 'sparse': sparse} if sparse else {'': dense},
        payload=point.payload,
    ))

client.upsert(collection_name='data_ds_bsc', points=points)
print(f'Uploaded {len(points)} points')

info = client.get_collection('data_ds_bsc')
print(f'Points count: {info.points_count}')
"

# 6. Verify
curl http://localhost:6333/collections/data_ds_bsc
```

### New Instance to the Same Directory

```
# 1. Replace old storage.sqlite with new one directly in folder

# 2. Start Qdrant
nohup ~/qdrant --config-path ~/qdrant_config/config.yaml \
  > ~/qdrant.log 2>&1 &
sleep 3
curl http://localhost:6333/collections

# 3. Verify
curl http://localhost:6333/collections/data_ds_bsc

# vLLM

## Start LLM

```
nohup vllm serve Qwen/Qwen3-4B-Instruct-2507-FP8 \
  --host 0.0.0.0 \
  --port 8000 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.28 \
  > ~/logs/vllm_llm.log 2>&1 &
```


```
tail -f ~/logs/vllm_llm.log
# Wait for: "Application startup complete"
# Ctrl+C to stop watching

```

```
# Verify
curl http://localhost:8000/v1/models
```

## Start embed

```
nohup vllm serve BAAI/bge-m3 \
  --host 0.0.0.0 \
  --port 8001 \
  --runner pooling \
  --gpu-memory-utilization 0.08 \
  > ~/logs/vllm_embed.log 2>&1 &
```


```
tail -f ~/logs/vllm_embed.log
# Wait for: "Application startup complete"
# Ctrl+C to stop watching

```

```
# Verify
curl http://localhost:8001/v1/models
```

## Start reranker

```
nohup vllm serve BAAI/bge-reranker-v2-m3 \
  --host 0.0.0.0 \
  --port 8002 \
  --runner pooling \
  --gpu-memory-utilization 0.08 \
  > ~/logs/vllm_rerank.log 2>&1 &
```

```
tail -f ~/logs/vllm_rerank.log
# Wait for: "Application startup complete"
# Ctrl+C to stop watching
```
```
# Verify
curl http://localhost:8002/v1/models
```

## Stop All vLLMs


```
pkill -f "vllm serve"
```

```
# Check usage:

nvidia-smi
```

`nvidia-smi`
- Bash command, that displays the status of NVIDIA GPUs, including GPU usage, memory usage, temperature, power consumption, driver/CUDA versions, and running GPU processes. Useful for monitoring and troubleshooting GPU workloads.

## Explanations to commands



`nohup ... &` — runs the process in the background and keeps it alive even if you close the terminal. Without this, vLLM dies the moment you close JupyterLab.

`vllm serve Qwen/Qwen3-4B-Instruct-2507-FP8` — this downloads the model from HuggingFace if it's not cached, then loads it into GPU VRAM and starts an HTTP server on port 8000.

- The model downloads to `~/.cache/huggingface/` — which on server is `/home/user1/.cache/huggingface/`.

`--host 0.0.0.0` - Accept connections from any IP, not just localhost - allows Python service to connect to it from within the container

`--port 8000` - Which port to listen on - matches LLM_BASE_URL in .env

`--max-model-len 8192` - Maximum total tokens per request (prompt + answer): enough for system prompt + chunks + history + answer. **Cuts KV cache by 4x vs 32K**

`--gpu-memory-utilization 0.28` - vram allocated to model

`> vllm.log 2>&1 &` — redirects all output to vllm.log, runs the process in the background so terminal stays free and the process survives closing JupyterLab.

`--` (double dash) - marks a named argument/flag for the program. The program reads these by name.

`>` (redirect) - shell operator - tells the shell where to send the output

# VENV and Requirements

```
cd /home/user1/rag-service-vllm
```

```
python3 -m venv /home/user1/.venv
```

```
. /home/user1/.venv/bin/activate
```

```
pip install -r requirements.txt
```

# Testing

From the same directory, where test_agent.py is placed;

tee is to see and save log at the same time
```
# In a second terminal:

python test_agent.py 2>&1 | tee ~/logs/run_test_agent.log
```

Save stable requirements:
```
pip freeze > requirements-rag-vllm-frozen.txt
```

# Compile .proto

```
cd /home/user1/rag-service-vllm/
python -m grpc_tools.protoc \
    -I. \
    --python_out=src/ \
    --grpc_python_out=src/ \
    rag_service.proto
```